### General settings

In [1]:
from google.colab import drive
import os
import sys

drive.mount('/content/gdrive/')

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/Colab Notebooks'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

#Import client
from millionaire_client import MillionaireClient, AuthenticationError

#Get password
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

#Login
API_URL = "http://131.175.15.22:51111/"
username = "Pastasciutta"
password = pwd
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/Colab Notebooks']

Welcome, Pastasciutta! (Role: student)


In [2]:
comp_id = 1 #entertainment, history, science, math

def play_game(game):
  # Play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      for opt in question.options:
        print(f"{opt.id}: {opt.text}")
      print()

      option_texts = [opt.text for opt in question.options]
      response_id, response_text = pick_answer(tokenizer, model, question.text, option_texts)
      print(f"Selected answer: {response_id} - {response_text}")

      result = game.answer(response_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")


# RAG model

First, we will set the model and its corresponding prompts.



In [3]:
!pip install -U transformers accelerate bitsandbytes
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

#quantize to 4 bits to make it "smaller"
quant_config = BitsAndBytesConfig(load_in_4bit=True)

#load model and tokenizer
model_id = "Qwen/Qwen2.5-14B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

This will be all the needed functions for picking the correct answer.

Important notes are that:

*   The model is given only one shot at querying wikipedia
*   The model can only give one query





## First version

In [4]:
import requests
import re

def build_query_prompt(question, options):
    system_prompt = { "role": "system",
    "content": """Generate a Wikipedia search query for answering a quiz question.

Rules:
- Search for the topic/entity in the question, not the answer.
- Include distinctive names, places, or events.
- Never output a single generic word.
- Output only the query."""}

    user_prompt = {"role": "user", "content": f"""Question: {question}

A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}"""}

    return [system_prompt, user_prompt]


def build_doc_prompt(question, options, documents):
    system_prompt = {"role": "system", "content": """You are an expert quiz player competing on Who Wants to Be a Millionaire.
You will receive a question, four labeled options (A, B, C, D), and a set of reference documents retrieved from Wikipedia.
Use the documents as your primary source of truth to determine the correct answer.

Respond with ONLY the letter of the correct answer: A, B, C, or D.
Do not explain your reasoning. Do not write anything else."""}

    formatted_docs = "\n\n".join(f"[Document {i+1}]\n{doc}" for i, doc in enumerate(documents))

    user_prompt = {"role": "user", "content": f"""Question: {question}

A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}

Reference Documents:
{formatted_docs}"""}

    return [system_prompt, user_prompt]

def pick_answer(tokenizer, model, question , options):
  #Ask the model
  model_prompt = build_query_prompt(question,options)

  print("Calling model")
  query = call_model(tokenizer, model, model_prompt)
  query = query.strip()

  print(f"The model wants to perform a document search with the following query: {query}")
  found_docs = wikisearch(query)
  print(f"The following documents have been found: {found_docs}")

  model_prompt = build_doc_prompt(question, options, found_docs)
  response = call_model(tokenizer, model, model_prompt)
  letter_to_index = {"A": 0, "B": 1, "C": 2, "D": 3}

  print(f"Model answered {response}")
  match = re.search(r"\b(A|B|C|D)\b", response)
  if match:
      letter = match.group(1)
      return letter_to_index[letter], options[letter_to_index[letter]]
  else:
      print("Model did not output valid response")
      return None, None

def call_model(tokenizer, model, prompt):
  inputs = tokenizer.apply_chat_template(prompt, return_tensors="pt", return_dict=True, add_generation_prompt=True).to("cuda")
  outputs = model.generate(**inputs, max_new_tokens=200, pad_token_id=tokenizer.eos_token_id)
  new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
  response = tokenizer.decode(new_tokens, skip_special_tokens=True)
  return response


HEADERS = {
    "User-Agent": "WhoWantsToBeAMillionaire-Bot/1.0 (research project; bianchigianpaolo2@gmail.com)"
}
def wikisearch(query):
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": 3
    }
    search_response = requests.get(search_url, params=search_params, headers=HEADERS)

    if not search_response.text or search_response.status_code != 200:
        print("Wikipedia search failed")
        return []

    titles = [r["title"] for r in search_response.json()["query"]["search"]]

    docs = []
    for title in titles:
        extract_params = {
            "action": "query",
            "titles": title,
            "prop": "extracts",
            "exintro": True,
            "explaintext": True,
            "format": "json"
        }
        extract_response = requests.get(search_url, params=extract_params, headers=HEADERS)

        if not extract_response.text or extract_response.status_code != 200:
            print(f"Failed to fetch extract for: {title}")
            continue

        pages = extract_response.json()["query"]["pages"]
        for page in pages.values():
            if "extract" in page:
                docs.append(page["extract"])
    return docs

## Second version

In [9]:
import requests
import re
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

HEADERS = {
    "User-Agent": "WhoWantsToBeAMillionaire-Bot/1.0 (research project; bianchigianpaolo2@gmail.com)"
}

def build_query_prompt(question, options):
    system_prompt = {"role": "system", "content": """#TASK: You are helping answer a Who Wants to Be a Millionaire question by generating Wikipedia search queries.

#RULES:
- Generate between 1 and 5 queries focused on the TOPIC and CONTEXT of the question, not the answer options.
- Never copy the options verbatim as queries — they are rarely good search terms.
- Think about what Wikipedia articles would contain the answer and search for those.
- Each query on a new line.
- No numbering, no bullets, no explanations, no quotes.
- Output only the queries.

#EXAMPLE:
Question: What term describes the founding of Carthage by the legendary Queen Dido?
Good queries:
Queen Dido founding of Carthage
Carthage history origin Phoenician
Dido legendary queen Carthage"""}

    user_prompt = {"role": "user", "content": f"""#QUESTION: {question}

#OPTIONS:
A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}"""}

    return [system_prompt, user_prompt]


def build_doc_prompt(question, options, documents):
    system_prompt = {"role": "system", "content": """#TASK:
You are an expert quiz player competing on Who Wants to Be a Millionaire.
You will receive a question, four labeled options (A, B, C, D), and a set of reference documents retrieved from Wikipedia.
Use the documents as your primary source of truth to determine the correct answer.

#RULES:
Respond with ONLY the letter of the correct answer: A, B, C, or D.
Before answering, think step by step about what the documents say. Then output only the letter.
Do not explain your reasoning. Do not write anything else."""}

    formatted_docs = "\n\n".join(f"[Document {i+1}]\n{doc}" for i, doc in enumerate(documents))

    user_prompt = {"role": "user", "content": f"""#QUESTION: {question}
#OPTIONS:
A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}

#REFERENCE DOCUMENTS:
{formatted_docs}"""}

    return [system_prompt, user_prompt]


def chunk_text(text, chunk_size=200):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks


def get_top_chunks(question, options, chunks, top_k=3):
    query_text = f"{question} {' '.join(options)}"
    question_embedding = embedder.encode(query_text)
    chunk_embeddings = embedder.encode(chunks)

    question_norm = question_embedding / np.linalg.norm(question_embedding)
    chunks_norm = chunk_embeddings / np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
    similarities = chunks_norm @ question_norm

    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [chunks[i] for i in top_indices]


from concurrent.futures import ThreadPoolExecutor

def wikisearch_multi(queries, question, options):
    """Run multiple queries in parallel and pool all chunks"""
    def search_one(query):
        query = query.strip().replace('"', '')
        if not query:
            return []
        return wikisearch_single(query)

    with ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(search_one, queries))

    # Pool all chunks from all queries
    all_chunks = [chunk for doc_chunks in results for chunk in doc_chunks]

    if not all_chunks:
        return []

    return get_top_chunks(question, options, all_chunks, top_k=3)


def wikisearch_single(query):
    """Fetch and chunk articles for a single query"""
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": 3
    }
    search_response = requests.get(search_url, params=search_params, headers=HEADERS)

    if not search_response.text or search_response.status_code != 200:
        return []

    titles = [r["title"] for r in search_response.json()["query"]["search"]]

    all_chunks = []
    for title in titles:
        extract_params = {
            "action": "query",
            "titles": title,
            "prop": "extracts",
            "explaintext": True,
            "format": "json"
        }
        extract_response = requests.get(search_url, params=extract_params, headers=HEADERS)

        if not extract_response.text or extract_response.status_code != 200:
            continue

        pages = extract_response.json()["query"]["pages"]
        for page in pages.values():
            if "extract" in page:
                all_chunks.extend(chunk_text(page["extract"]))

    return all_chunks


def pick_answer(tokenizer, model, question, options):
    # Step 1: generate multiple search queries
    model_prompt = build_query_prompt(question, options)
    print("Generating search queries...")
    raw_queries = call_model(tokenizer, model, model_prompt).strip()
    queries = [q.strip() for q in raw_queries.split("\n") if q.strip()]
    print(f"Queries: {queries}")

    # Step 2: fetch in parallel, pool and rank chunks
    found_docs = wikisearch_multi(queries, question, options)
    print(f"Retrieved {len(found_docs)} chunks")

    # Step 3: answer with documents
    model_prompt = build_doc_prompt(question, options, found_docs)
    print("Answering with documents...")
    response = call_model(tokenizer, model, model_prompt)
    print(f"Model answered: {response}")

    letter_to_index = {"A": 0, "B": 1, "C": 2, "D": 3}
    match = re.search(r"\b(A|B|C|D)\b", response)
    if match:
        letter = match.group(1)
        return letter_to_index[letter], options[letter_to_index[letter]]
    else:
        print("Model did not output valid response")
        return None, None

def call_model(tokenizer, model, prompt):
    inputs = tokenizer.apply_chat_template(
        prompt,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    ).to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        pad_token_id=tokenizer.eos_token_id
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Confidence version

In [6]:
import re
import torch
import torch.nn.functional as F

letter_to_index = {"A": 0, "B": 1, "C": 2, "D": 3}


# -------------------------
# Prompts
# -------------------------
def build_question_prompt(question, options):
    return [
        {
            "role": "system",
            "content": (
                "You answer multiple-choice questions.\n"
                "Respond ONLY with A, B, C, or D."
            )
        },
        {
            "role": "user",
            "content": f"""Question:
{question}

A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}"""
        }
    ]


def build_query_prompt(question, options):
    return [
        {
            "role": "system",
            "content": (
                "Generate a Wikipedia search query.\n"
                "- Focus on key entities in the question\n"
                "- Output ONLY the query\n"
                "- Avoid answering the question"
            )
        },
        {
            "role": "user",
            "content": f"""Question: {question}

A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}"""
        }
    ]


def build_doc_prompt(question, options, docs):
    formatted = "\n\n".join(f"[Doc {i+1}]\n{d}" for i, d in enumerate(docs))

    return [
        {
            "role": "system",
            "content": (
                "You are a fact-checking QA system.\n"
                "Use ONLY the provided documents.\n"
                "Return ONLY A, B, C, or D."
            )
        },
        {
            "role": "user",
            "content": f"""Question:
{question}

A) {options[0]}
B) {options[1]}
C) {options[2]}
D) {options[3]}

Documents:
{formatted}"""
        }
    ]


# -------------------------
# Main pipeline
# -------------------------
def pick_answer(tokenizer, model, question, options):

    # -------------------------
    # 1. PRIOR (no evidence)
    # -------------------------
    prior_prompt = build_question_prompt(question, options)
    prior_answer = call_model(tokenizer, model, prior_prompt).strip()

    print(f"[PRIOR] {prior_answer}")

    # -------------------------
    # 2. QUERY GENERATION + RETRIEVAL
    # -------------------------
    query_prompt = build_query_prompt(question, options)

    docs = []
    seen = set()

    for _ in range(3):
        query = call_model(tokenizer, model, query_prompt).strip()

        print(f"[QUERY] {query}")

        results = wikisearch(query)

        for d in results:
            if d not in seen:
                seen.add(d)
                docs.append(d)

    # -------------------------
    # 3. EVIDENCE ANSWER
    # -------------------------
    if docs:
        evidence_prompt = build_doc_prompt(question, options, docs)
        evidence_answer = call_model(tokenizer, model, evidence_prompt).strip()
        print(f"[EVIDENCE] {evidence_answer}")
    else:
        evidence_answer = None
        print("[EVIDENCE] no documents")

    # -------------------------
    # 4. DECISION LOGIC
    # -------------------------

    def normalize(ans):
        m = re.search(r"[ABCD]", ans)
        return m.group(0) if m else None

    prior_letter = normalize(prior_answer)
    evidence_letter = normalize(evidence_answer) if evidence_answer else None

    # CASE 1: no evidence
    if evidence_letter is None:
        if prior_letter:
            return letter_to_index[prior_letter], options[letter_to_index[prior_letter]]
        return None, None

    # CASE 2: agreement
    if prior_letter == evidence_letter:
        return letter_to_index[evidence_letter], options[letter_to_index[evidence_letter]]

    # CASE 3: disagreement → trust evidence
    return letter_to_index[evidence_letter], options[letter_to_index[evidence_letter]]

def call_model(tokenizer, model, prompt):
  inputs = tokenizer.apply_chat_template(prompt, return_tensors="pt", return_dict=True, add_generation_prompt=True).to("cuda")
  outputs = model.generate(**inputs, max_new_tokens=200, pad_token_id=tokenizer.eos_token_id)
  new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
  response = tokenizer.decode(new_tokens, skip_special_tokens=True)
  return response

def answer_with_confidence(tokenizer, model, prompt):
    inputs = tokenizer.apply_chat_template(
        prompt,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    ).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1]
    probs = F.softmax(logits, dim=-1)
    choices = ["A", "B", "C", "D"]
    scores = {}
    for c in choices:
        token_id = tokenizer.encode(c, add_special_tokens=False)[0]
        scores[c] = probs[token_id].item()
    best = max(scores, key=scores.get)
    confidence = scores[best]
    return best, confidence, scores


HEADERS = {
    "User-Agent": "WhoWantsToBeAMillionaire-Bot/1.0 (research project; bianchigianpaolo2@gmail.com)"
}
def wikisearch(query):
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": 3
    }
    search_response = requests.get(search_url, params=search_params, headers=HEADERS)

    if not search_response.text or search_response.status_code != 200:
        print("Wikipedia search failed")
        return []

    titles = [r["title"] for r in search_response.json()["query"]["search"]]

    docs = []
    for title in titles:
        extract_params = {
            "action": "query",
            "titles": title,
            "prop": "extracts",
            "exintro": True,
            "explaintext": True,
            "format": "json"
        }
        extract_response = requests.get(search_url, params=extract_params, headers=HEADERS)

        if not extract_response.text or extract_response.status_code != 200:
            print(f"Failed to fetch extract for: {title}")
            continue

        pages = extract_response.json()["query"]["pages"]
        for page in pages.values():
            if "extract" in page:
                docs.append(page["extract"])
    return docs

# Run game

In [11]:
# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")
print()
play_game(game)


=== Starting Game ===
Session ID: 44616
Total number of questions: 15


--- Level 1 ---
Q: Which period in ancient Egyptian history is characterized by the unification of Upper and Lower Egypt under a single ruler?
0: The Early Dynastic Period
1: The New Kingdom
2: The Old Kingdom
3: The First Intermediate Period

Generating search queries...
Queries: ['unification of Upper and Lower Egypt', 'Egypt unification first pharaoh', 'Early Dynastic Period Egypt unification']
Retrieved 3 chunks
Answering with documents...
Model answered: A
Selected answer: 0 - The Early Dynastic Period
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
Q: What was the primary economic activity that contributed to the prosperity of ancient Greek city-states?
0: Trade
1: Agriculture
2: Slavery
3: Manufacturing

Generating search queries...
Queries: ['ancient Greek city-states economy', 'Greek city-states trade agriculture', 'economic activities ancient Greece prosperity', 'Greek city-states primary industries']

Are we serious?

The question was "Q: What is the fundamental principle behind the naming of Athens according to modern scholars?

0: The name was derived from the first olive tree planted in Athens.

1: The name reflects the city's patronage by Athena, the goddess of wisdom.

2: The goddess Athena took her name from the city.

3: The name comes from the word 'flower' or 'flowering city'."

Wikipedia extract: "According to Greek mythology, the city was named after Athena, the ancient Greek goddess of wisdom, but modern scholars generally agree that the goddess took her name after the city". But 2 is incorrect according to the game